# ReLive Knowledge Base Test Harness

This notebook tests the full write/read workflow:

1. Load sample texts from `data/`
2. Write extracted knowledge into Neo4j using the writer graph
3. Run chat-like read queries against the knowledge base

## Multi-Concept Extraction

The writer now extracts **multiple distinct concepts** from each text. If a text contains multiple:
- Environments
- Problems  
- Solutions
- Results

...each combination is extracted as a separate knowledge entry. For example, a text describing 3 different solutions creates 3 separate knowledge entries in the database.

## Provider Configuration

- **LLM**: Choose from `"openai"`, `"gemini"`, `"inception"` (Mercury 2), or `"none"` for deterministic local mode
- **Embedder**: Choose from `"openai"`, `"gemini"`, or `"qwen"` (local fallback)
  - **Qwen3 Embedding** automatically downloads on first use (no API key needed)
  - Falls back from cloud providers if not configured

## Neo4j Configuration

- **Target**: Set `NEO4J_TARGET` in the setup cell to `"local"`, `"hosted"`, or `"docker"`
- **Hosted (Aura)**: Add to `.env`:
  - `NEO4J_HOSTED_NAME` — instance id (e.g. `97f8db06`)
  - `NEO4J_HOSTED_PASSWORD` — database password
  - Optional: `NEO4J_HOSTED_URI`, `NEO4J_HOSTED_USERNAME`, `NEO4J_HOSTED_DATABASE`

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from src.logic.orchestrator import AgentOrchestrator
from src.ai.providers import (
    GeminiEmbedderClient,
    GeminiLLMClient,
    GeminiProviderConfig,
    OpenAIEmbedderClient,
    OpenAILLMClient,
    OpenAIProviderConfig,
    InceptionEmbedderClient,
    InceptionLLMClient,
    InceptionProviderConfig,
    QwenEmbedderClient,
    QwenProviderConfig,
)
from src.ai.embedder_utils import get_embedder_safe

/home/r2/.pyenv/versions/relive_env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
# Choose provider: "none", "openai", "gemini", "inception" (LLM), or "qwen" (embeddings only).
PROVIDER = "inception"
EMBEDDER_PROVIDER = "qwen"  # Use "qwen" for local embeddings, or "openai"/"gemini"

# Neo4j: "local" (Docker/localhost), "hosted" (Aura / remote), or in-memory debug.
NEO4J_TARGET = "hosted"  # "local" | "hosted" | "docker"
USE_IN_MEMORY_DEBUG = False

import importlib
import os
from pathlib import Path

from dotenv import load_dotenv

# Load .env from project root (Jupyter cwd is often not the repo root)
_PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists() or (candidate / "src" / "database").is_dir():
        _PROJECT_ROOT = candidate
        break
load_dotenv(_PROJECT_ROOT / ".env", override=False)

if USE_IN_MEMORY_DEBUG:
    os.environ["NEO4J_DEBUG_IN_MEMORY"] = "true"
else:
    os.environ["NEO4J_TARGET"] = NEO4J_TARGET

# Reload modules in case notebook kernel cached older code.
import src.database.config as db_config
import src.database.infrastructure.driver as neo_driver
import src.database.manager as db_manager
import src.logic.orchestrator as orchestrator_mod
importlib.reload(db_config)
importlib.reload(neo_driver)
importlib.reload(db_manager)
importlib.reload(orchestrator_mod)
neo_driver.Neo4jDriver._instance = None

llm = None
embedder = None

# Setup LLM provider
if PROVIDER == "openai":
    cfg = OpenAIProviderConfig.from_env()
    llm = OpenAILLMClient(cfg)
elif PROVIDER == "gemini":
    cfg = GeminiProviderConfig.from_env()
    llm = GeminiLLMClient(cfg)
elif PROVIDER == "inception":
    cfg = InceptionProviderConfig()
    llm = InceptionLLMClient(cfg)

# Setup Embedder provider (with fallback to Qwen if not configured)
if EMBEDDER_PROVIDER == "openai":
    if llm is None or not isinstance(llm, OpenAILLMClient):
        cfg = OpenAIProviderConfig.from_env()
    embedder = OpenAIEmbedderClient(cfg)
elif EMBEDDER_PROVIDER == "gemini":
    if llm is None or not isinstance(llm, GeminiLLMClient):
        cfg = GeminiProviderConfig.from_env()
    embedder = GeminiEmbedderClient(cfg)
elif EMBEDDER_PROVIDER == "qwen":
    cfg = QwenProviderConfig()
    embedder = QwenEmbedderClient(cfg)
else:
    # Default: use safe fallback (tries None first, then Qwen)
    embedder = get_embedder_safe(primary_embedder=None, fallback_to_qwen=True)

from src.database.config import Neo4jSettings, load_project_env
from src.database.manager import DatabaseManager
load_project_env(override=False)
neo4j_settings = Neo4jSettings()
if neo4j_settings.target == "hosted" and not neo4j_settings.credentials_ok_for_hosted():
    raise RuntimeError(
        "Hosted Neo4j password not loaded. Check .env has NEO4J_HOSTED_PASSWORD "
        f"and restart the kernel. Project root: {_PROJECT_ROOT}"
    )
orchestrator = AgentOrchestrator(
    llm=llm,
    embedder=embedder,
    db=DatabaseManager(settings=neo4j_settings),
)
await orchestrator.initialize()
print("Orchestrator initialized")
print(f"  LLM Provider: {PROVIDER}")
print(f"  Embedder Provider: {EMBEDDER_PROVIDER}")
print(f"  Neo4j: {neo4j_settings.connection_summary()}")
print(f"  Debug Mode: {USE_IN_MEMORY_DEBUG}")

Orchestrator initialized
  LLM Provider: inception
  Embedder Provider: qwen
  Neo4j: hosted (neo4j+s://…7477e481.databases.neo4j.io, db=7477e481, user=7477e481, creds=ok)
  Debug Mode: False


In [4]:
# Diagnostic: Test LLM and Embedder provider connectivity
print("🔍 Testing Providers...\n")

# Test LLM
print("LLM Provider:")
if llm is None:
    print("  ⚠️  No LLM provider configured")
    print("     The system will use heuristic fallbacks for extraction/reflection")
else:
    try:
        test_prompt = "Respond with exactly: {\"test\": true}"
        test_response = await llm.ainvoke(test_prompt)
        print(f"  ✓ LLM responding ({len(test_response)} chars)")
        if not test_response.strip():
            print("  ⚠️  WARNING: LLM returned empty response!")
    except Exception as e:
        print(f"  ✗ LLM error: {e}")

# Test Embedder
print("\nEmbedding Provider:")
if embedder is None:
    print("  ✗ No embedder available!")
else:
    try:
        test_embedding = await embedder.embed("test")
        print(f"  ✓ Embedder responding ({len(test_embedding)}-dim)")
    except Exception as e:
        print(f"  ✗ Embedder error: {e}")

🔍 Testing Providers...

LLM Provider:
  ✓ LLM responding (14 chars)

Embedding Provider:
Loading Qwen3 Embedding model from: /home/r2/Documents/Projects/ReLived/models/qwen
Model: Qwen/Qwen3-Embedding-0.6B
Device: cuda


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✓ Model loaded successfully on cuda
  ✓ Embedder responding (1024-dim)


In [5]:
data_dir = Path("data")
files = sorted(data_dir.glob("*.txt"))
print("Sample files:")
for f in files:
    print("-", f)

texts = [{"name": f.name, "text": f.read_text(encoding="utf-8")} for f in files]
print(f"Loaded {len(texts)} text documents")

Sample files:
- data/ai_problems.txt
- data/electronics_motor_overheat.txt
- data/python_etl_memory_spike.txt
- data/survival_cold_weather_fire.txt
- data/survival_guide.txt
Loaded 5 text documents


In [6]:
ingest_results = []
for item in texts:
    result = await orchestrator.run(mode="write", text=item["text"], environment_hint="auto")
    # Now result contains multiple persisted entries (list) instead of one
    persisted_list = result.get("persisted", [])
    ingest_results.append({
        "file": item["name"],
        "num_concepts_extracted": len(persisted_list),
        "knowledge_entry_ids": [entry.get("knowledge_entry_id") for entry in persisted_list],
        "errors": result.get("errors", []),
    })

print(json.dumps(ingest_results, indent=2))

[
  {
    "file": "ai_problems.txt",
    "num_concepts_extracted": 9,
    "knowledge_entry_ids": [
      "f3e81cdd-77a9-4507-81cb-35be4ec35472",
      "45662ff5-13ce-4676-8e9d-ffacc6365082",
      "1e49b576-b84c-4657-b92a-c43ca259158b",
      "b8b2fc77-473e-4801-b37f-48a67fc9185d",
      "c5a1c879-d6fb-447e-a015-70ed8fb10b21",
      "94917255-80a3-4758-8da0-42dfe8811714",
      "327e5eda-8a2c-4513-91e1-fc98e2151150",
      "2d6d09b5-a6d7-4b65-9b0d-24e66b86dce5",
      "c6eebd3c-c671-445c-88a5-1ba5f4313c1f"
    ],
    "errors": []
  },
  {
    "file": "electronics_motor_overheat.txt",
    "num_concepts_extracted": 1,
    "knowledge_entry_ids": [
      "cbe63e40-bf38-441c-9ce2-80abd9f3e9bb"
    ],
    "errors": []
  },
  {
    "file": "python_etl_memory_spike.txt",
    "num_concepts_extracted": 1,
    "knowledge_entry_ids": [
      "0e631d6a-c537-44dd-b9fa-2d7d4ee5a6c6"
    ],
    "errors": []
  },
  {
    "file": "survival_cold_weather_fire.txt",
    "num_concepts_extracted": 1,
    "kn

In [7]:
async def chat_turn(user_message: str, top_k: int | None = None) -> dict:
    """Run a single chat-like retrieval turn against the knowledge base."""
    response = await orchestrator.run(mode="read", text=user_message, top_k=top_k)
    return response.get("response", response)


def pretty_print_turn(query: str, payload: dict) -> None:
    print(f"USER: {query}\n")
    ranked = payload.get("ranked_results", [])
    if not ranked:
        print("ASSISTANT: No close matches found.")
        return

    best = ranked[0]
    print("ASSISTANT: Top match")
    print(json.dumps(best, indent=2))
    print("\nALTERNATIVES:")
    print(json.dumps(payload.get("alternatives", []), indent=2))

In [8]:
from pyvis.network import Network

neo4j_db = orchestrator.db.repository.nodes.get_graph_snapshot()

def visualize_debug_db(nodes, relationships):
    net = Network(notebook=True, directed=True, aggregation=True)
    
    # Add nodes from your in-memory backend
    for node in nodes:
        net.add_node(node.id, label=str(node.labels), title=str(node.properties))
        
    # Add relationships
    for rel in relationships:
        net.add_edge(rel.start_node, rel.end_node, label=rel.type)
        
    return net.show("debug_graph.html")

ModuleNotFoundError: No module named 'pyvis'

In [9]:
visualize_debug_db()

NameError: name 'visualize_debug_db' is not defined

In [10]:
# Example chat-style conversation turns
queries = [
    # "How can I reduce overheating in a dusty drone motor setup?",
    # "How can I light a fire?",
    "lighting fire with sun",
    # "Tropical rainforest",
]

for q in queries:
    payload = await chat_turn(q)  # top_k omitted -> adaptive traversal mode
    pretty_print_turn(q, payload)
    print("\n" + "=" * 80 + "\n")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `ALTERNATIVE_SOLUTION` does not exist in database `7477e481`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        MATCH (n)-[:ALTERNATIVE_SOLUTION|ALTERNATIVE_RESULT|LEADS_TO_RESULT|WORKS_IN_ENVIRONMENT*1..2]-(alt)\n        WHERE elementId(n) = $node_id\n        RETURN elementId(alt) AS alternative_id,\n               labels(alt) AS labels,\n               coalesce(alt.text, '') AS text

USER: lighting fire with sun

ASSISTANT: Top match
{
  "node_id": "4:7085b40b-0fd7-4c4e-95b7-e6639e30d787:81",
  "text": "Use a magnifying glass or polished soda\u2011can base as a reflective lens to focus sunlight onto charred cloth or dried dung, creating a focused point of heat that ignites an ember",
  "avg_similarity": 0.8858529329299927,
  "concept_coverage": 1,
  "score": 0.6800970530509949,
  "concepts_matched": [
    "solution"
  ]
}

ALTERNATIVES:
[
  {
    "source_node_id": "4:7085b40b-0fd7-4c4e-95b7-e6639e30d787:81",
    "alternatives": [
      {
        "alternative_id": "4:7085b40b-0fd7-4c4e-95b7-e6639e30d787:83",
        "labels": [
          "Result"
        ],
        "text": "A slow\u2011creeping ember that spreads to dry grasses, providing fire with minimal fuel and energy expenditure; limitation: ineffective during dust storms or low\u2011sun angles"
      },
      {
        "alternative_id": "4:7085b40b-0fd7-4c4e-95b7-e6639e30d787:79",
        "labels": [
         

In [11]:
# Optional: manual interactive loop in notebook output
# Run this cell repeatedly with new prompt values.
user_query = "suggest alternatives for drone overheating in hot environment"
payload = await chat_turn(user_query)
pretty_print_turn(user_query, payload)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `ALTERNATIVE_SOLUTION` does not exist in database `7477e481`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        MATCH (n)-[:ALTERNATIVE_SOLUTION|ALTERNATIVE_RESULT|LEADS_TO_RESULT|WORKS_IN_ENVIRONMENT*1..2]-(alt)\n        WHERE elementId(n) = $node_id\n        RETURN elementId(alt) AS alternative_id,\n               labels(alt) AS labels,\n               coalesce(alt.text, '') AS text

USER: suggest alternatives for drone overheating in hot environment

ASSISTANT: Top match
{
  "node_id": "4:7085b40b-0fd7-4c4e-95b7-e6639e30d787:56",
  "text": "overheating causing protective throttling and occasional emergency stop",
  "avg_similarity": 0.8699884414672852,
  "concept_coverage": 1,
  "score": 0.6689919090270995,
  "concepts_matched": [
    "problem"
  ]
}

ALTERNATIVES:
[
  {
    "source_node_id": "4:7085b40b-0fd7-4c4e-95b7-e6639e30d787:55",
    "alternatives": [
      {
        "alternative_id": "4:7085b40b-0fd7-4c4e-95b7-e6639e30d787:57",
        "labels": [
          "Solution"
        ],
        "text": "add vent filters, tune PWM duty-cycle ramping, and introduce firmware temperature smoothing"
      },
      {
        "alternative_id": "4:7085b40b-0fd7-4c4e-95b7-e6639e30d787:59",
        "labels": [
          "Result"
        ],
        "text": "37% drop in thermal cutoff events, longer continuous flight windows, slight increase in maintenance time for filter cle

In [12]:
# Cleanup when done (recommended)
await orchestrator.close()
print("Orchestrator closed")

Orchestrator closed


# Database Visualization

## Neo4j Database Inspector & Visualizer

Connect to the Docker Neo4j instance and visualize the knowledge graph.

In [13]:
import subprocess
from neo4j import AsyncGraphDatabase
import asyncio

async def check_docker_neo4j():
    """Check if Docker Neo4j is available."""
    try:
        # Try to connect to Docker Neo4j on localhost
        driver = AsyncGraphDatabase.driver(
            "bolt://localhost:7687",
            auth=("neo4j", "password"),
            connection_timeout=5.0
        )
        async with driver.session() as session:
            result = await session.run("RETURN 1")
            await driver.close()
        return True, "Connected to Docker Neo4j ✓"
    except Exception as e:
        return False, f"Docker Neo4j not available: {str(e)}"

def start_docker_neo4j():
    """Start Neo4j via docker-compose."""
    try:
        result = subprocess.run(
            ["sudo", "docker-compose", "up", "-d", "neo4j"],
            capture_output=True,
            text=True,
            timeout=30
        )
        if result.returncode == 0:
            return True, "Docker Compose started Neo4j ✓"
        else:
            return False, f"Docker Compose error: {result.stderr}"
    except FileNotFoundError:
        return False, "docker-compose not found. Install Docker Desktop or run: sudo docker-compose up -d"
    except Exception as e:
        return False, f"Error starting Docker: {str(e)}"

# Check and start
available, msg = await check_docker_neo4j()
print(msg)

if not available:
    print("\nAttempting to start Docker Neo4j...")
    started, msg = start_docker_neo4j()
    print(msg)
    
    if started:
        print("Waiting for Neo4j to be ready...")
        await asyncio.sleep(5)
        available, msg = await check_docker_neo4j()
        print(msg)


Docker Neo4j not available: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [Errno 111] Connect call failed ('::1', 7687, 0, 0))
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connect call failed ('127.0.0.1', 7687))

Attempting to start Docker Neo4j...
Docker Compose error: sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: a password is required



In [ ]:
async def fetch_graph_data(driver):
    """Fetch all nodes and relationships from Neo4j."""
    async with driver.session() as session:
        # Fetch nodes with their labels and properties
        nodes_result = await session.run("""
            MATCH (n)
            RETURN id(n) as node_id, labels(n) as labels, properties(n) as props
            LIMIT 1000
        """)
        nodes = await nodes_result.data()
        
        # Fetch relationships
        rels_result = await session.run("""
            MATCH (a)-[r]->(b)
            RETURN id(a) as source_id, id(b) as target_id, type(r) as rel_type, 
                   properties(r) as rel_props
            LIMIT 2000
        """)
        rels = await rels_result.data()
    
    return nodes, rels

async def get_graph_stats(driver):
    """Get statistics about the Neo4j database."""
    async with driver.session() as session:
        node_count = await session.run("MATCH (n) RETURN count(n) as cnt")
        node_data = await node_count.data()
        
        rel_count = await session.run("MATCH ()-[r]->() RETURN count(r) as cnt")
        rel_data = await rel_count.data()
        
        labels = await session.run("CALL db.labels() YIELD label RETURN label")
        label_data = await labels.data()
    
    return {
        "nodes": node_data[0]["cnt"] if node_data else 0,
        "relationships": rel_data[0]["cnt"] if rel_data else 0,
        "labels": [row["label"] for row in label_data] if label_data else []
    }

# Connect to Neo4j on localhost
docker_driver = AsyncGraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password")
)

# Fetch data
nodes, relationships = await fetch_graph_data(docker_driver)
stats = await get_graph_stats(docker_driver)

print(f"📊 Database Statistics:")
print(f"  Nodes: {stats['nodes']}")
print(f"  Relationships: {stats['relationships']}")
print(f"  Node Labels: {', '.join(stats['labels']) if stats['labels'] else 'None yet'}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=3, column=20, offset=42>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 42, 'line': 3, 'column': 20}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (n)\n            RETURN id(n) as node_id, labels(n) as labels, properties(n) as props\n            LIMIT 1000\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced 

📊 Database Statistics:
  Nodes: 26
  Relationships: 49
  Node Labels: Node, Document, KnowledgeEntry, Environment, Problem, Solution, Mechanism, Result


In [ ]:
from pyvis.network import Network
import json

def colorize_node(labels):
    """Assign color based on node label."""
    color_map = {
        "Environment": "#FF6B6B",
        "Problem": "#4ECDC4", 
        "Solution": "#45B7D1",
        "Mechanism": "#FFA07A",
        "Result": "#98D8C8",
        "KnowledgeEntry": "#FFD93D",
        "Reference": "#9B59B6",
    }
    if labels:
        primary_label = labels[0]
        return color_map.get(primary_label, "#95E1D3")
    return "#95E1D3"

def create_graph_visualization(nodes, relationships):
    """Create an interactive pyvis network visualization."""
    net = Network(
        height="750px",
        width="100%",
        directed=True,
        notebook=True,
        physics=True
    )
    
    # Configure physics
    net.toggle_physics(True)
    net.show_buttons()
    
    # Create node id mapping
    node_map = {}
    for node in nodes:
        node_id = node["node_id"]
        node_map[node_id] = node
        
        labels = node["labels"]
        props = node["props"]
        
        # Create label
        display_label = f"{labels[0] if labels else 'Node'}"
        
        # Add useful property info to title
        title_info = json.dumps(props, indent=2, default=str)
        
        color = colorize_node(labels)
        
        net.add_node(
            node_id,
            label=display_label,
            title=title_info,
            color=color,
            size=30,
            shape="dot"
        )
    
    # Add relationships
    for rel in relationships:
        source = rel["source_id"]
        target = rel["target_id"]
        rel_type = rel["rel_type"]
        
        # Only add if both nodes exist
        if source in node_map and target in node_map:
            net.add_edge(
                source,
                target,
                label=rel_type,
                title=json.dumps(rel["rel_props"], indent=2, default=str),
                color="#888888",
                arrows="to"
            )
    
    return net

# Generate visualization
if nodes:
    net = create_graph_visualization(nodes, relationships)
    net.show("neo4j_knowledge_graph.html")
    print("✓ Graph visualization saved as 'neo4j_knowledge_graph.html'")
    print("  Open in browser to explore and move nodes interactively")
else:
    print("ℹ️ No data in database yet. Run the write workflow first to populate the graph.")


ModuleNotFoundError: No module named 'pyvis'

In [ ]:
async def query_database(cypher_query):
    """Execute a custom Cypher query against Neo4j."""
    try:
        async with docker_driver.session() as session:
            result = await session.run(cypher_query)
            data = await result.data()
        return data
    except Exception as e:
        return f"Error: {str(e)}"

async def get_node_details(node_id):
    """Get detailed information about a specific node."""
    query = f"MATCH (n) WHERE id(n) = {node_id} RETURN n, labels(n) as labels"
    
    async with docker_driver.session() as session:
        result = await session.run(query)
        data = await result.data()
    
    if data:
        node = data[0]
        print(f"Node ID: {node_id}")
        print(f"Labels: {', '.join(node['labels'])}")
        print(f"Properties: {json.dumps(node['n'], indent=2, default=str)}")
        
        # Get connected relationships
        rel_query = f"""
            MATCH (n)-[r]-(m) WHERE id(n) = {node_id}
            RETURN type(r) as rel_type, labels(m) as target_labels, 
                   properties(m) as target_props, id(m) as target_id
        """
        rel_result = await docker_driver.session().run(rel_query)
        rels = await rel_result.data()
        
        if rels:
            print(f"\nConnected Nodes ({len(rels)}):")
            for rel in rels:
                print(f"  --[{rel['rel_type']}]--> {rel['target_labels'][0]}")
    else:
        print(f"Node {node_id} not found")

# Example: Query for knowledge entries
print("📝 Recent Knowledge Entries:")
knowledge_entries = await query_database("""
    MATCH (ke:KnowledgeEntry)
    RETURN ke.id, ke.created_at, ke.source_title
    ORDER BY ke.created_at DESC
    LIMIT 5
""")

for entry in knowledge_entries:
    print(f"  - {entry}")


In [ ]:
# Cleanup Neo4j driver when done
await docker_driver.close()
print("✓ Neo4j driver closed")

# Optional: Stop Docker container
def stop_docker_neo4j():
    """Stop Neo4j via docker-compose (keeps data in volumes)."""
    try:
        result = subprocess.run(
            ["docker-compose", "down"],
            capture_output=True,
            text=True,
            timeout=30
        )
        if result.returncode == 0:
            print("✓ Docker Compose stopped (data persisted in volumes)")
        else:
            print(f"Note: {result.stderr}")
    except Exception as e:
        print(f"Note: {str(e)}")

# Uncomment to stop containers:
# stop_docker_neo4j()
